# Домашняя работа: Deep Q-Network и улучшения

Этот ноутбук предназначен для самостоятельной проработки материалов из конспекта [`note_10_deep_q_network.md`](../../notes/md/note_10_deep_q_network.md).
Вы реализуете ключевые элементы Deep Q-Network для среды **LunarLander-v2**, а затем закрепите улучшения Double и Dueling.


## Учебные цели
- реализовать базовый DQN с реплей буфером и target сети для среды `LunarLander-v2`
- сравнить две схемы epsilon-жадной политики: линейную и экспоненциальную
- внедрить Double DQN и Dueling архитектуру и проанализировать их влияние на обучение
- сформулировать наблюдения и рекомендации по настройке DQN


## Как выполнять работу
- Проходите ноутбук сверху вниз; каждый блок TODO нужно закрыть собственным кодом.
- Если работаете в Google Colab, сперва выполните установку зависимостей (см. следующий блок).
- Сохраняйте промежуточные результаты и скриншоты графиков для отчета.
- В конце ноутбука заполните секцию с выводами.


## Полезные советы

### О среде LunarLander-v2:
- **Задача**: посадить лунный модуль между двумя флагами
- **Наблюдения**: 8 непрерывных значений (позиция, скорость, угол, контакт с землей)
- **Действия**: 4 дискретных действия (ничего, левый двигатель, главный двигатель, правый двигатель)
- **Награды**: +100 за успешную посадку, штрафы за использование двигателей и краш
- **Решена**: средняя награда ≥ 200 за 100 последовательных эпизодов
- **Сложность**: сложнее CartPole, требует ~50k-100k шагов для обучения

### Рекомендуемый порядок работы:
1. Сначала убедитесь, что ReplayBuffer работает корректно (уже реализован)
2. Проверьте расписания epsilon на графике
3. Реализуйте и протестируйте функции потерь
4. Запустите базовый DQN на малом числе шагов (10k) для отладки
5. Только после этого запускайте полные эксперименты (50k+ шагов)

### Особенности DQN для LunarLander:
- **Warmup phase**: первые 5k шагов просто заполняют буфер, обучение не идет
- **Target network**: обновляется реже, чем policy сеть (это ключ к стабильности)
- **Epsilon decay**: должен закончиться примерно на 70-80% от total_steps
- **Batch size**: обычно 64-128 для LunarLander
- **Buffer**: минимум 50k переходов для стабильного обучения

### Подготовка окружения
Запустите ячейку ниже только при необходимости (например, в Colab).


In [ ]:
# Если запускаете ноутбук в Colab, раскомментируйте строки ниже.
# !pip install gymnasium[box2d] torch torchvision matplotlib tqdm -q
# Примечание: LunarLander требует установки box2d (для физики)

In [1]:
import math
import random
from collections import deque, namedtuple
from dataclasses import dataclass
from typing import Callable, Deque, Dict, Iterable, List, Optional, Tuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


### Знакомство с LunarLander-v2

Перед началом работы полезно понаблюдать за средой:

In [ ]:
# Исследование среды LunarLander-v2
env = gym.make("LunarLander-v2")

print("=" * 60)
print("LunarLander-v2 Environment Info")
print("=" * 60)

print(f"\nObservation Space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Low: {env.observation_space.low}")
print(f"  High: {env.observation_space.high}")

print(f"\nAction Space: {env.action_space}")
print(f"  Number of actions: {env.action_space.n}")
print("\nAction meanings:")
print("  0 - do nothing")
print("  1 - fire left orientation engine")
print("  2 - fire main engine")
print("  3 - fire right orientation engine")

# Запустим один случайный эпизод для примера
state, info = env.reset(seed=SEED)
print(f"\nInitial state: {state}")
print("  [0-1]: x, y position")
print("  [2-3]: x, y velocity")
print("  [4]: angle")
print("  [5]: angular velocity")
print("  [6-7]: left leg contact, right leg contact")

total_reward = 0
steps = 0
done = False

print("\nRunning random policy for one episode...")
while not done and steps < 1000:
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

print(f"\nRandom policy results:")
print(f"  Total reward: {total_reward:.2f}")
print(f"  Episode length: {steps} steps")
print(f"  Terminated: {terminated}")
print(f"  Final state: {state}")

env.close()

print("\n" + "=" * 60)
print("Note: LunarLander is considered solved when average reward ≥ 200")
print("=" * 60)

## 1. Реплей буфер и переходы
Реализуйте буфер воспроизведения, который хранит последние `capacity` переходов и умеет:

1. добавлять новый переход с перезаписью старых;
2. семплировать случайный минибатч и возвращать тензоры PyTorch.

Используйте именованный кортеж `Transition` и храните `done` как float (0.0 или 1.0).


In [2]:
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))


class ReplayBuffer:
    '''Простой циклический буфер для DQN.'''

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.buffer: Deque[Transition] = deque(maxlen=capacity)

    def push(self, transition: Transition) -> None:
        '''Добавляет переход в буфер с перезаписью старых элементов.'''
        self.buffer.append(transition)

    def sample(self, batch_size: int) -> Transition:
        '''Возвращает батч, где каждое поле представлено тензором на устройстве device.'''
        batch = Transition(*zip(*random.sample(self.buffer, batch_size)))
        states = torch.as_tensor(np.stack(batch.state), dtype=torch.float32, device=device)
        actions = torch.as_tensor(batch.action, dtype=torch.int64, device=device)
        rewards = torch.as_tensor(batch.reward, dtype=torch.float32, device=device)
        next_states = torch.as_tensor(np.stack(batch.next_state), dtype=torch.float32, device=device)
        dones = torch.as_tensor(batch.done, dtype=torch.float32, device=device)
        return Transition(states, actions, rewards, next_states, dones)

    def __len__(self) -> int:
        return len(self.buffer)


### Тестирование ReplayBuffer

Проверим, что буфер работает корректно:

In [ ]:
# Тест ReplayBuffer
test_buffer = ReplayBuffer(capacity=100)

# Добавим несколько переходов
for i in range(5):
    transition = Transition(
        state=np.array([i, i+1, i+2, i+3], dtype=np.float32),
        action=i % 2,
        reward=float(i * 0.1),
        next_state=np.array([i+1, i+2, i+3, i+4], dtype=np.float32),
        done=float(i == 4),
    )
    test_buffer.push(transition)

print(f"Размер буфера: {len(test_buffer)}")

# Семплируем батч
if len(test_buffer) >= 3:
    batch = test_buffer.sample(batch_size=3)
    print(f"States shape: {batch.state.shape}, device: {batch.state.device}")
    print(f"Actions shape: {batch.action.shape}")
    print(f"Rewards: {batch.reward}")
    print(f"Dones: {batch.done}")
    print("✅ ReplayBuffer работает корректно!")
else:
    print("⚠️ Недостаточно переходов для семплирования")

## 2. Epsilon-жадная политика и расписания
Сначала реализуйте две функции расписания epsilon: линейную и экспоненциальную. Затем напишите стратегию выбора действия.

- Линейное расписание должно уменьшать epsilon от `start` до `end` за `duration` шагов.
- Экспоненциальное расписание: epsilon(t) = epsilon_min + (epsilon_max - epsilon_min) * exp(-t / tau).
- Функция выбора действия возвращает индекс действия int.


In [3]:
class LinearSchedule:
    def __init__(self, start: float, end: float, duration: int):
        self.start = start
        self.end = end
        self.duration = max(1, duration)

    def __call__(self, step: int) -> float:
        '''Возвращает значение epsilon для данного шага.'''
        progress = min(step / self.duration, 1.0)
        return self.start + (self.end - self.start) * progress


class ExponentialSchedule:
    def __init__(self, start: float, end: float, tau: float):
        self.start = start
        self.end = end
        self.tau = max(1e-6, tau)

    def __call__(self, step: int) -> float:
        '''Возвращает значение epsilon по экспоненциальной формуле.'''
        return self.end + (self.start - self.end) * math.exp(-step / self.tau)


def epsilon_greedy_action(policy_net: nn.Module, state: np.ndarray, epsilon: float, action_space: gym.spaces.Discrete) -> int:
    '''Возвращает действие согласно текущему epsilon-гридди правилу.'''
    if random.random() < epsilon:
        return int(action_space.sample())
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        q_values = policy_net(state_tensor)
    return int(torch.argmax(q_values, dim=1).item())


### Проверка расписаний epsilon

Визуализируйте расписания перед использованием:

In [ ]:
# Проверка расписаний epsilon
import matplotlib.pyplot as plt

steps = np.arange(0, 20000)

# Линейное расписание
linear_schedule = LinearSchedule(start=1.0, end=0.01, duration=15000)
linear_epsilons = [linear_schedule(s) for s in steps]

# Экспоненциальное расписание
exp_schedule = ExponentialSchedule(start=1.0, end=0.01, tau=5000)
exp_epsilons = [exp_schedule(s) for s in steps]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(steps, linear_epsilons, label='Linear')
plt.xlabel('Step')
plt.ylabel('Epsilon')
plt.title('Linear Schedule')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(steps, exp_epsilons, label='Exponential', color='orange')
plt.xlabel('Step')
plt.ylabel('Epsilon')
plt.title('Exponential Schedule')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

print(f"Linear: epsilon at step 0={linear_schedule(0):.3f}, step 15000={linear_schedule(15000):.3f}")
print(f"Exponential: epsilon at step 0={exp_schedule(0):.3f}, step 15000={exp_schedule(15000):.3f}")

## 3. Q-сеть и target сеть
Для среды `CartPole-v1` достаточно MLP. Реализуйте прямой проход сети и инициализацию весов Ксавье.

После реализации создайте отдельную target сеть и функцию ее обновления.


In [4]:
def init_layer(layer: nn.Linear) -> None:
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)


class DQN(nn.Module):
    def __init__(self, observation_dim: int, action_dim: int, hidden_dims: Tuple[int, ...] = (128, 128)):
        super().__init__()
        dims = (observation_dim,) + hidden_dims
        layers: List[nn.Module] = []
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            lin = nn.Linear(in_dim, out_dim)
            init_layer(lin)
            layers.extend([lin, nn.ReLU()])
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(hidden_dims[-1], action_dim)
        init_layer(self.head)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Прямой проход через backbone и выходной слой.'''
        if x.dim() == 1:
            x = x.unsqueeze(0)
        features = self.backbone(x.float())
        return self.head(features)


### Тестирование DQN сети

Проверим форму выходов и инициализацию:

In [ ]:
# Тест DQN сети
test_env = gym.make("LunarLander-v2")
obs_dim = test_env.observation_space.shape[0]
act_dim = test_env.action_space.n
test_env.close()

print(f"LunarLander-v2:")
print(f"  Observation dim: {obs_dim}")
print(f"  Action dim: {act_dim}")
print(f"  Observation space: {test_env.observation_space}")
print(f"  Action meanings: 0=nothing, 1=fire left, 2=fire main, 3=fire right")

# Создаём сеть
test_dqn = DQN(obs_dim, act_dim, hidden_dims=(128, 128)).to(device)
print(f"\nАрхитектура сети:\n{test_dqn}")

# Тест прямого прохода
test_state = torch.randn(1, obs_dim, device=device)
with torch.no_grad():
    q_values = test_dqn(test_state)

print(f"\nВход: {test_state.shape}")
print(f"Выход (Q-values): {q_values.shape}")
print(f"Q-values для 4 действий: {q_values}")

# Проверка выбора действия
best_action = torch.argmax(q_values, dim=1).item()
print(f"\nЛучшее действие: {best_action}")

# Проверка батча
batch_states = torch.randn(32, obs_dim, device=device)
with torch.no_grad():
    batch_q_values = test_dqn(batch_states)
print(f"\nБатч вход: {batch_states.shape}")
print(f"Батч выход: {batch_q_values.shape}")

print("\n✅ DQN сеть работает корректно!")

In [5]:
def hard_update(target: nn.Module, source: nn.Module) -> None:
    '''Копирует веса из source в target (жесткое обновление).'''
    target.load_state_dict(source.state_dict())


def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    '''Плавное обновление target сети при tau между 0 и 1.'''
    with torch.no_grad():
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.mul_(1 - tau).add_(tau * param.data)


## 4. Потери DQN и цикл обучения
Заполните функции ниже. Они отвечают за расчет TD ошибки и основной цикл обучения.

1. `compute_td_target` считает r + gamma * max_a Q_target(s_next, a).
2. `compute_dqn_loss` возвращает SmoothL1Loss между Q(s,a) и таргетом.
3. `train_dqn` собирает опыт, обновляет буфер, делает шаги оптимизатора и логирует метрики.

Для логирования используйте словарь списков (`Dict[str, List[float]]`).


In [6]:
@torch.no_grad()
def compute_td_target(target_net: nn.Module, next_states: torch.Tensor, rewards: torch.Tensor, dones: torch.Tensor, gamma: float) -> torch.Tensor:
    '''Базовая DQN цель: r + gamma * max_a Q_target(s_next, a).'''
    max_next_q = target_net(next_states).max(dim=1).values
    return rewards + gamma * (1.0 - dones) * max_next_q


def compute_dqn_loss(
    policy_net: nn.Module,
    target_net: nn.Module,
    batch: Transition,
    gamma: float,
    double_dqn: bool = False,
) -> torch.Tensor:
    '''Рассчитывает SmoothL1Loss между Q(s,a) и таргетами.'''
    current_q = policy_net(batch.state).gather(1, batch.action.long().unsqueeze(1)).squeeze(1)
    if double_dqn:
        target_q = compute_double_dqn_target(policy_net, target_net, batch.next_state, batch.reward, batch.done, gamma)
    else:
        target_q = compute_td_target(target_net, batch.next_state, batch.reward, batch.done, gamma)
    target_q = target_q.detach()
    return F.smooth_l1_loss(current_q, target_q)


def train_dqn(
    env: gym.Env,
    policy_net: nn.Module,
    target_net: nn.Module,
    optimizer: optim.Optimizer,
    buffer: ReplayBuffer,
    schedule: Callable[[int], float],
    total_steps: int,
    warmup_steps: int,
    batch_size: int,
    gamma: float,
    update_target_every: int,
    double_dqn: bool = False,
) -> Dict[str, List[float]]:
    '''Запускает обучение DQN и возвращает логи с метриками.'''
    policy_net.train()
    target_net.eval()
    hard_update(target_net, policy_net)

    metrics: Dict[str, List[float]] = {
        "episode_reward": [],
        "episode_length": [],
        "epsilon": [],
        "loss": [],
    }

    state, _ = env.reset(seed=SEED)
    state = np.asarray(state, dtype=np.float32)
    episode_reward = 0.0
    episode_length = 0

    for step in range(1, total_steps + 1):
        epsilon = float(schedule(step - 1))
        action = epsilon_greedy_action(policy_net, state, epsilon, env.action_space)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        next_state_arr = np.asarray(next_state, dtype=np.float32)
        buffer.push(
            Transition(
                state=np.asarray(state, dtype=np.float32),
                action=action,
                reward=float(reward),
                next_state=next_state_arr,
                done=float(done),
            )
        )

        if step >= warmup_steps and len(buffer) >= batch_size:
            batch = buffer.sample(batch_size)
            optimizer.zero_grad()
            loss = compute_dqn_loss(policy_net, target_net, batch, gamma, double_dqn=double_dqn)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=10.0)
            optimizer.step()
            metrics["loss"].append(loss.item())

        if step % update_target_every == 0:
            hard_update(target_net, policy_net)

        episode_reward += float(reward)
        episode_length += 1
        state = next_state_arr

        if done:
            metrics["episode_reward"].append(episode_reward)
            metrics["episode_length"].append(episode_length)
            metrics["epsilon"].append(epsilon)
            state, _ = env.reset()
            state = np.asarray(state, dtype=np.float32)
            episode_reward = 0.0
            episode_length = 0

    return metrics


## 5. Оценка политики
Реализуйте детерминированную оценку для полученной политики (без epsilon рандомизации).


In [7]:
@torch.no_grad()
def evaluate_policy(env: gym.Env, policy_net: nn.Module, episodes: int = 5) -> float:
    '''Возвращает среднюю награду по заданному числу эпизодов для жадной политики.'''
    policy_net.eval()
    rewards = []
    for _ in range(episodes):
        state, _ = env.reset()
        episode_reward = 0.0
        done = False
        while not done:
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_values = policy_net(state_tensor)
            action = int(torch.argmax(q_values, dim=1).item())
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += float(reward)
            state = next_state
            done = terminated or truncated
        rewards.append(episode_reward)
    policy_net.train()
    return float(np.mean(rewards))


## 6. Эксперимент A - сравнение расписаний
1. Создайте две копии среды и сетей с одинаковыми начальными весами.
2. Обучите DQN с линейным и экспоненциальным расписанием (по 20k-40k шагов).
3. Постройте графики `episode_reward` и `epsilon` (используйте `matplotlib`).
4. Заполните текстовые выводы ниже.

Заполните ячейки кода и markdown, отмеченные TODO.


In [ ]:
# TODO: настройте гиперпараметры, окружения и расписания
# Подсказки для LunarLander-v2:
# 1. Создайте среду LunarLander-v2
# 2. Определите размерности: observation_dim=8 и action_dim=4
# 3. Создайте две пары сетей (policy + target) с одинаковыми начальными весами
# 4. Определите гиперпараметры:
#    - total_steps: 80000-120000 (LunarLander требует больше шагов, чем CartPole)
#    - warmup_steps: 5000-10000
#    - batch_size: 64-128
#    - gamma: 0.99
#    - learning_rate: 5e-4 (меньше, чем для CartPole)
#    - buffer_capacity: 50000-100000 (больше для стабильности)
#    - update_target_every: 1000-2000

# Пример структуры:
# env_name = "LunarLander-v2"
# env_linear = gym.make(env_name)
# env_exp = gym.make(env_name)
#
# observation_dim = env_linear.observation_space.shape[0]  # 8
# action_dim = env_linear.action_space.n  # 4
#
# # Гиперпараметры для LunarLander
# total_steps = 100000
# warmup_steps = 5000
# batch_size = 128
# gamma = 0.99
# lr = 5e-4
# buffer_capacity = 50000
# update_target_every = 1000
#
# # Сети для линейного расписания
# policy_net_linear = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_linear = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_linear.load_state_dict(policy_net_linear.state_dict())
#
# # Сети для экспоненциального расписания (с теми же начальными весами)
# policy_net_exp = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# policy_net_exp.load_state_dict(policy_net_linear.state_dict())
# target_net_exp = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_exp.load_state_dict(policy_net_linear.state_dict())
#
# # Оптимизаторы
# optimizer_linear = optim.Adam(policy_net_linear.parameters(), lr=lr)
# optimizer_exp = optim.Adam(policy_net_exp.parameters(), lr=lr)
#
# # Буферы
# buffer_linear = ReplayBuffer(buffer_capacity)
# buffer_exp = ReplayBuffer(buffer_capacity)
#
# # Расписания
# schedule_linear = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps))
# schedule_exp = ExponentialSchedule(start=1.0, end=0.01, tau=total_steps / 5)

raise NotImplementedError("TODO: подготовьте окружения, сети и расписания для эксперимента A")

NotImplementedError: TODO: подготовьте окружения, сети и расписания для эксперимента A

In [ ]:
# TODO: запустите обучение для линейного расписания
# Подсказки:
# 1. Используйте функцию train_dqn() с линейными сетями и расписанием
# 2. Сохраните результат в переменную metrics_linear
# 3. Оцените финальную политику через evaluate_policy()

# Пример:
# print("Обучение с линейным расписанием epsilon...")
# metrics_linear = train_dqn(
#     env=env_linear,
#     policy_net=policy_net_linear,
#     target_net=target_net_linear,
#     optimizer=optimizer_linear,
#     buffer=buffer_linear,
#     schedule=schedule_linear,
#     total_steps=total_steps,
#     warmup_steps=warmup_steps,
#     batch_size=batch_size,
#     gamma=gamma,
#     update_target_every=update_target_every,
#     double_dqn=False,
# )
#
# eval_reward_linear = evaluate_policy(env_linear, policy_net_linear, episodes=10)
# print(f"Линейное расписание: средняя награда при оценке = {eval_reward_linear:.1f}")
# env_linear.close()

raise NotImplementedError("TODO: запустите train_dqn с линейным расписанием")

In [ ]:
# TODO: запустите обучение для экспоненциального расписания
# Подсказка: аналогично предыдущей ячейке, но с экспоненциальными объектами

# Пример:
# print("Обучение с экспоненциальным расписанием epsilon...")
# metrics_exp = train_dqn(
#     env=env_exp,
#     policy_net=policy_net_exp,
#     target_net=target_net_exp,
#     optimizer=optimizer_exp,
#     buffer=buffer_exp,
#     schedule=schedule_exp,
#     total_steps=total_steps,
#     warmup_steps=warmup_steps,
#     batch_size=batch_size,
#     gamma=gamma,
#     update_target_every=update_target_every,
#     double_dqn=False,
# )
#
# eval_reward_exp = evaluate_policy(env_exp, policy_net_exp, episodes=10)
# print(f"Экспоненциальное расписание: средняя награда при оценке = {eval_reward_exp:.1f}")
# env_exp.close()

raise NotImplementedError("TODO: запустите train_dqn с экспоненциальным расписанием")

In [ ]:
# TODO: визуализируйте сравнение наград и epsilon
# Подсказки:
# 1. Создайте фигуру с 3 графиками: episode_reward, epsilon, loss
# 2. Используйте скользящее среднее для сглаживания наград
# 3. Добавьте легенду с финальными оценками

# Вспомогательная функция для скользящего среднего
def moving_average(values: List[float], window: int = 20) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if arr.size < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='valid')

# Пример визуализации:
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#
# # График 1: Награды
# ma_linear = moving_average(metrics_linear['episode_reward'], window=20)
# ma_exp = moving_average(metrics_exp['episode_reward'], window=20)
# axes[0].plot(ma_linear, label=f'Linear (eval={eval_reward_linear:.0f})')
# axes[0].plot(ma_exp, label=f'Exponential (eval={eval_reward_exp:.0f})')
# axes[0].set_title('Episode Rewards (MA 20)')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward')
# axes[0].legend()
# axes[0].grid(True)
#
# # График 2: Epsilon
# axes[1].plot(metrics_linear['epsilon'], label='Linear', alpha=0.7)
# axes[1].plot(metrics_exp['epsilon'], label='Exponential', alpha=0.7)
# axes[1].set_title('Epsilon Schedule')
# axes[1].set_xlabel('Episode')
# axes[1].set_ylabel('Epsilon')
# axes[1].legend()
# axes[1].grid(True)
#
# # График 3: Loss
# if len(metrics_linear['loss']) > 0:
#     ma_loss_linear = moving_average(metrics_linear['loss'], window=100)
#     ma_loss_exp = moving_average(metrics_exp['loss'], window=100)
#     axes[2].plot(ma_loss_linear, label='Linear')
#     axes[2].plot(ma_loss_exp, label='Exponential')
#     axes[2].set_title('Training Loss (MA 100)')
#     axes[2].set_xlabel('Update Step')
#     axes[2].set_ylabel('Loss')
#     axes[2].legend()
#     axes[2].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: постройте графики для эксперимента A")

### Выводы по эксперименту A

**Ответьте на следующие вопросы:**

1. **TODO** Какое расписание epsilon привело к более быстрому росту наград в LunarLander?

2. **TODO** Какая схема показала меньше колебаний в наградах? (LunarLander имеет высокую дисперсию наград)

3. **TODO** Сравните eval_reward для обоих вариантов. Достигли ли вы порога решения (≥200)?

4. **TODO** Объясните, как форма кривой epsilon влияет на баланс exploration/exploitation в LunarLander.

5. **TODO** Какое расписание вы бы выбрали для LunarLander и почему?

## 7. Эксперимент B - Double DQN
Модифицируйте расчет таргета, чтобы использовать формулу Double DQN. Затем повторите обучение (на коротком бюджете шагов) и сравните распределение TD ошибки.

1. Реализуйте функцию `compute_double_dqn_target` ниже.
2. Измените `compute_dqn_loss`, чтобы принимать флаг `double_dqn` (при необходимости создайте новую функцию).
3. Перезапустите обучение и сохраните метрики.


In [ ]:
@torch.no_grad()
def compute_double_dqn_target(
    policy_net: nn.Module,
    target_net: nn.Module,
    next_states: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    '''Вычисляет таргет Double DQN, разделяя выбор и оценку действий.'''
    next_actions = policy_net(next_states).argmax(dim=1, keepdim=True)
    next_q = target_net(next_states).gather(1, next_actions).squeeze(1)
    return rewards + gamma * (1.0 - dones) * next_q


In [ ]:
def compute_loss_with_variant(
    policy_net: nn.Module,
    target_net: nn.Module,
    batch: Transition,
    gamma: float,
    mode: str = "vanilla",
) -> torch.Tensor:
    '''Хелпер для выбора варианта расчёта таргетов (vanilla или double).'''
    use_double = mode.lower() == "double"
    return compute_dqn_loss(policy_net, target_net, batch, gamma, double_dqn=use_double)


In [ ]:
# TODO: запустите обучение с Double DQN и сравните метрики
# Подсказки:
# 1. Создайте новые среды и сети для честного сравнения
# 2. Запустите два обучения: одно с double_dqn=False, другое с double_dqn=True
# 3. Используйте одинаковые гиперпараметры и расписание epsilon
# 4. Сравните награды и стабильность обучения
# 5. Для LunarLander Double DQN может значительно уменьшить переоценку Q-values

# Пример структуры:
# print("=" * 50)
# print("Эксперимент B: Vanilla DQN vs Double DQN на LunarLander")
# print("=" * 50)
#
# # Настройка для обоих экспериментов
# total_steps_b = 80000
# warmup_steps_b = 5000
# batch_size_b = 128
# gamma_b = 0.99
# lr_b = 5e-4
# buffer_capacity_b = 50000
# update_target_every_b = 1000
#
# # Vanilla DQN
# env_vanilla = gym.make("LunarLander-v2")
# obs_dim = env_vanilla.observation_space.shape[0]
# act_dim = env_vanilla.action_space.n
#
# policy_net_vanilla = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_vanilla = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_vanilla.load_state_dict(policy_net_vanilla.state_dict())
# optimizer_vanilla = optim.Adam(policy_net_vanilla.parameters(), lr=lr_b)
# buffer_vanilla = ReplayBuffer(buffer_capacity_b)
# schedule_vanilla = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_b))
#
# print("\n1. Обучение Vanilla DQN...")
# metrics_vanilla = train_dqn(
#     env=env_vanilla,
#     policy_net=policy_net_vanilla,
#     target_net=target_net_vanilla,
#     optimizer=optimizer_vanilla,
#     buffer=buffer_vanilla,
#     schedule=schedule_vanilla,
#     total_steps=total_steps_b,
#     warmup_steps=warmup_steps_b,
#     batch_size=batch_size_b,
#     gamma=gamma_b,
#     update_target_every=update_target_every_b,
#     double_dqn=False,
# )
# eval_vanilla = evaluate_policy(env_vanilla, policy_net_vanilla, episodes=10)
# print(f"Vanilla DQN: eval_reward = {eval_vanilla:.1f}")
# env_vanilla.close()
#
# # Double DQN
# env_double = gym.make("LunarLander-v2")
# policy_net_double = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# policy_net_double.load_state_dict(policy_net_vanilla.state_dict())  # Одинаковые начальные веса
# target_net_double = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_double.load_state_dict(policy_net_vanilla.state_dict())
# optimizer_double = optim.Adam(policy_net_double.parameters(), lr=lr_b)
# buffer_double = ReplayBuffer(buffer_capacity_b)
# schedule_double = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_b))
#
# print("\n2. Обучение Double DQN...")
# metrics_double = train_dqn(
#     env=env_double,
#     policy_net=policy_net_double,
#     target_net=target_net_double,
#     optimizer=optimizer_double,
#     buffer=buffer_double,
#     schedule=schedule_double,
#     total_steps=total_steps_b,
#     warmup_steps=warmup_steps_b,
#     batch_size=batch_size_b,
#     gamma=gamma_b,
#     update_target_every=update_target_every_b,
#     double_dqn=True,
# )
# eval_double = evaluate_policy(env_double, policy_net_double, episodes=10)
# print(f"Double DQN: eval_reward = {eval_double:.1f}")
# env_double.close()
#
# # Визуализация сравнения
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# ma_vanilla = moving_average(metrics_vanilla['episode_reward'], window=50)
# ma_double = moving_average(metrics_double['episode_reward'], window=50)
#
# axes[0].plot(ma_vanilla, label=f'Vanilla DQN (eval={eval_vanilla:.0f})', alpha=0.8)
# axes[0].plot(ma_double, label=f'Double DQN (eval={eval_double:.0f})', alpha=0.8)
# axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Solved threshold')
# axes[0].set_title('LunarLander: Episode Rewards')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward (MA 50)')
# axes[0].legend()
# axes[0].grid(True)
#
# if len(metrics_vanilla['loss']) > 0:
#     ma_loss_vanilla = moving_average(metrics_vanilla['loss'], window=100)
#     ma_loss_double = moving_average(metrics_double['loss'], window=100)
#     axes[1].plot(ma_loss_vanilla, label='Vanilla DQN', alpha=0.8)
#     axes[1].plot(ma_loss_double, label='Double DQN', alpha=0.8)
#     axes[1].set_title('Training Loss (MA 100)')
#     axes[1].set_xlabel('Update Step')
#     axes[1].set_ylabel('Loss')
#     axes[1].legend()
#     axes[1].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: сравните базовый DQN и Double DQN")

### Выводы по эксперименту B

**Ответьте на следующие вопросы:**

1. **TODO** Заметили ли вы разницу в стабильности обучения между Vanilla и Double DQN?

2. **TODO** Какой метод быстрее достиг хороших результатов?

3. **TODO** Сравните eval_reward обоих вариантов.

4. **TODO** Как различаются кривые loss для Vanilla и Double DQN?

5. **TODO** В каких задачах Double DQN дает наибольший выигрыш?

## 8. Эксперимент C - Dueling Architecture
Реализуйте Dueling сеть и сравните динамику обучения.

1. Реализуйте класс `DuelingDQN` ниже.
2. Сравните с базовым DQN на том же расписании.
3. Сделайте вывод, дает ли ускорение или стабильность.


In [ ]:
class DuelingDQN(nn.Module):
    def __init__(self, observation_dim: int, action_dim: int, hidden_dims: Tuple[int, ...] = (128, 128)):
        super().__init__()
        dims = (observation_dim,) + hidden_dims
        layers: List[nn.Module] = []
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            lin = nn.Linear(in_dim, out_dim)
            init_layer(lin)
            layers.extend([lin, nn.ReLU()])
        self.feature_extractor = nn.Sequential(*layers)
        self.value_head = nn.Linear(hidden_dims[-1], 1)
        self.adv_head = nn.Linear(hidden_dims[-1], action_dim)
        init_layer(self.value_head)
        init_layer(self.adv_head)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Возвращает Q(s,a) = V(s) + A(s,a) - mean(A(s,a)).'''
        if x.dim() == 1:
            x = x.unsqueeze(0)
        features = self.feature_extractor(x.float())
        value = self.value_head(features)
        advantage = self.adv_head(features)
        advantage = advantage - advantage.mean(dim=1, keepdim=True)
        return value + advantage


In [ ]:
# TODO: обучите Dueling DQN и сравните результаты с базовым
# Подсказки:
# 1. Используйте DuelingDQN вместо обычного DQN
# 2. Все остальные параметры оставьте такими же, как в базовом DQN
# 3. Сравните скорость обучения и стабильность
# 4. Для LunarLander Dueling может помочь лучше оценивать состояния

# Пример структуры:
# print("=" * 50)
# print("Эксперимент C: Standard DQN vs Dueling DQN на LunarLander")
# print("=" * 50)
#
# # Гиперпараметры
# total_steps_c = 80000
# warmup_steps_c = 5000
# batch_size_c = 128
# gamma_c = 0.99
# lr_c = 5e-4
# buffer_capacity_c = 50000
# update_target_every_c = 1000
#
# # Standard DQN
# env_standard = gym.make("LunarLander-v2")
# obs_dim = env_standard.observation_space.shape[0]
# act_dim = env_standard.action_space.n
#
# policy_net_standard = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_standard = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_standard.load_state_dict(policy_net_standard.state_dict())
# optimizer_standard = optim.Adam(policy_net_standard.parameters(), lr=lr_c)
# buffer_standard = ReplayBuffer(buffer_capacity_c)
# schedule_standard = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_c))
#
# print("\n1. Обучение Standard DQN...")
# metrics_standard = train_dqn(
#     env=env_standard,
#     policy_net=policy_net_standard,
#     target_net=target_net_standard,
#     optimizer=optimizer_standard,
#     buffer=buffer_standard,
#     schedule=schedule_standard,
#     total_steps=total_steps_c,
#     warmup_steps=warmup_steps_c,
#     batch_size=batch_size_c,
#     gamma=gamma_c,
#     update_target_every=update_target_every_c,
#     double_dqn=False,
# )
# eval_standard = evaluate_policy(env_standard, policy_net_standard, episodes=10)
# print(f"Standard DQN: eval_reward = {eval_standard:.1f}")
# env_standard.close()
#
# # Dueling DQN
# env_dueling = gym.make("LunarLander-v2")
# policy_net_dueling = DuelingDQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_dueling = DuelingDQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_dueling.load_state_dict(policy_net_dueling.state_dict())
# optimizer_dueling = optim.Adam(policy_net_dueling.parameters(), lr=lr_c)
# buffer_dueling = ReplayBuffer(buffer_capacity_c)
# schedule_dueling = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_c))
#
# print("\n2. Обучение Dueling DQN...")
# metrics_dueling = train_dqn(
#     env=env_dueling,
#     policy_net=policy_net_dueling,
#     target_net=target_net_dueling,
#     optimizer=optimizer_dueling,
#     buffer=buffer_dueling,
#     schedule=schedule_dueling,
#     total_steps=total_steps_c,
#     warmup_steps=warmup_steps_c,
#     batch_size=batch_size_c,
#     gamma=gamma_c,
#     update_target_every=update_target_every_c,
#     double_dqn=False,
# )
# eval_dueling = evaluate_policy(env_dueling, policy_net_dueling, episodes=10)
# print(f"Dueling DQN: eval_reward = {eval_dueling:.1f}")
# env_dueling.close()
#
# # Визуализация
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# ma_standard = moving_average(metrics_standard['episode_reward'], window=50)
# ma_dueling = moving_average(metrics_dueling['episode_reward'], window=50)
#
# axes[0].plot(ma_standard, label=f'Standard DQN (eval={eval_standard:.0f})', alpha=0.8)
# axes[0].plot(ma_dueling, label=f'Dueling DQN (eval={eval_dueling:.0f})', alpha=0.8)
# axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Solved threshold')
# axes[0].set_title('LunarLander: Standard vs Dueling')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward (MA 50)')
# axes[0].legend()
# axes[0].grid(True)
#
# if len(metrics_standard['loss']) > 0:
#     ma_loss_standard = moving_average(metrics_standard['loss'], window=100)
#     ma_loss_dueling = moving_average(metrics_dueling['loss'], window=100)
#     axes[1].plot(ma_loss_standard, label='Standard DQN', alpha=0.8)
#     axes[1].plot(ma_loss_dueling, label='Dueling DQN', alpha=0.8)
#     axes[1].set_title('Training Loss (MA 100)')
#     axes[1].set_xlabel('Update Step')
#     axes[1].set_ylabel('Loss')
#     axes[1].legend()
#     axes[1].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: обучите Dueling DQN и сравните с MLP")

### Выводы по эксперименту C

**Ответьте на следующие вопросы:**

1. **TODO** Как Dueling архитектура повлияла на обучение?

2. **TODO** Dueling DQN учится быстрее или медленнее?

3. **TODO** Сравните дисперсию наград между Standard и Dueling.

4. **TODO** Какая архитектура показала лучший eval_reward?

5. **TODO** В каких сценариях Dueling архитектура наиболее полезна?

## 9. Вопросы для самопроверки

1. **TODO** Объясните, почему replay buffer критически важен для DQN. Что произойдет, если обучаться на последовательных переходах?
2. **TODO** Зачем нужна отдельная target network? Почему нельзя использовать одну сеть для Q(s,a) и для таргета?
3. **TODO** Когда Double DQN дает наибольший выигрыш? Приведите примеры задач.
4. **TODO** Объясните интуицию за разделением Q(s,a) = V(s) + A(s,a) в dueling architecture.
5. **TODO** Какие гиперпараметры оказались наиболее чувствительными в ваших экспериментах?
6. **TODO** Какой сигнал вы отслеживали для определения момента остановки обучения?
7. **TODO** Для каких задач DQN не подходит? Когда стоит использовать policy gradient методы?
